# 14. Missing and invalid values
Missingness has causes. A blank age can mean not collected, declined, or lost during processing. Filling a value does not recover the true measurement; it provides a consistent representation for a model. Invalid values such as impossible ages require domain checks before imputation. Indicators can preserve whether a value was originally absent.

Inspect missingness by feature and subgroup. Fit an imputer on training data only, ideally inside a pipeline. SimpleImputer can use a mean, median, most-frequent value, or constant. KNNImputer borrows values from nearby rows and needs more care with feature scales and computation. Check the policy for columns that are entirely missing.

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer
train = np.array([[1, 10], [3, np.nan], [5, 30]]) # Training examples with one missing value.
new = np.array([[np.nan, 50]]) # A future example.
imputer = SimpleImputer(strategy="median", add_indicator=True) # Learn medians and missingness markers.
imputer.fit(train) # Learn from training examples only.
print("Medians:", imputer.statistics_) # Replacement value per original feature.
print("Transformed new row:", imputer.transform(new)) # Apply learned values without refitting.
assert np.isfinite(imputer.transform(new)).all() # Check finite numeric output.

## Compare imputers and preserve missingness
A median is a simple, auditable reference. KNNImputer borrows from rows close in their observed values, so scale and meaningful distances matter. IterativeImputer models a missing column using other columns repeatedly; it is experimental and requires an explicit enable import in this frozen version. It does not recover missing ground truth or solve missing-not-at-random mechanisms. MissingIndicator can explicitly record missingness; features="all" creates an indicator for every input column so new missingness does not change width. Combine indicators with imputed values, rather than treating indicator-only output as the original features.

In [ ]:
import numpy as np
from sklearn.experimental import enable_iterative_imputer # Explicit experimental opt-in before importing the class.
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer, MissingIndicator
train = np.array([[1., 10.], [2., np.nan], [3., 30.], [4., 40.], [5., 50.]]) # Correlated demonstration features.
new = np.array([[np.nan, 25.], [3.5, np.nan]]) # Two future rows with different missingness.
for imputer in [SimpleImputer(strategy="median"), KNNImputer(n_neighbors=2), IterativeImputer(max_iter=20, random_state=42)]:
    transformed = imputer.fit(train).transform(new) # Only training data teaches the imputation rule.
    print(type(imputer).__name__, transformed.round(2).tolist()) # Compare assumptions through their output.
    assert transformed.shape == new.shape and np.isfinite(transformed).all() # Check the representation.
flags = MissingIndicator(features="all").fit(train).transform(new) # Stable indicator width for all columns.
print("Missing flags:", flags.astype(int).tolist()) # One indicates an originally absent value.

### Reading the result
Different filled values reflect different assumptions. This deliberately tiny example compares behavior, not predictive accuracy. An indicator value of one records absence, while the imputed measurement is an estimate. Evaluate the complete imputer-plus-model pipeline on suitable development folds before choosing a method.

### Transfer exercise
Compare median and KNN imputation inside a scaled pipeline on a documented development dataset. Record runtime as well as score. Investigate whether missingness itself is predictive.

## Your turn
Compare mean and median imputation after adding an extreme training value. Explain why neither method should fit on the final test set.

Try the task before opening the solution on the course website.